In [1]:
import networkx as nx
import numpy as np
import random
import pandas as pd
from collections import Counter

# 1. グラフ G が未定義の場合、再生成する (NameError回避)
try:
    G
except NameError:
    print("[Info] グラフ G を再生成しています...")
    n = 5000
    tau1 = 3.0
    tau2 = 1.016
    mu = 0.8
    average_degree = 10
    min_community = 100
    max_community = 1500
    G = nx.LFR_benchmark_graph(n, tau1, tau2, mu, average_degree=average_degree, min_community=min_community, max_community=max_community, seed=6)
    G = nx.Graph(G)
    G.remove_edges_from(nx.selfloop_edges(G))

# 2. コミュニティの抽出
raw_communities = {frozenset(G.nodes[v]['community']) for v in G}
communities = [list(c) for c in raw_communities]

# 3. 実験の前提設定（虚偽コミュニティ: ID 1、自陣: ID 2）
opponent_id = 1
target_id = 2
opponent_nodes = communities[opponent_id]
target_nodes = communities[target_id]
sub_G_opp = G.subgraph(opponent_nodes)
inter_edges = list(nx.edge_boundary(G, target_nodes, opponent_nodes))

print(f"--- 🚀 シナリオ別シードノード選出 (ターゲット: {target_id} vs 相手: {opponent_id}) ---")

# 【シナリオ1】ブリッジ：相手国側のノード v のみ
opp_side_nodes = [v for u, v in inter_edges]
seed_scenario_1 = [Counter(opp_side_nodes).most_common(1)[0][0]]
print(f"【シナリオ①】ブリッジ(敵陣側ハブ): {seed_scenario_1}")

# 【シナリオ2】ブリッジ：自陣側のノード u のみ
target_side_nodes = [u for u, v in inter_edges]
seed_scenario_2 = [Counter(target_side_nodes).most_common(1)[0][0]]
print(f"【シナリオ②】ブリッジ(自陣側裏切り者): {seed_scenario_2}")

# 【シナリオ3】ブリッジ：両方のノード u と v を同時
seed_scenario_3 = [seed_scenario_1[0], seed_scenario_2[0]]
print(f"【シナリオ③】ブリッジ(国境両側同時): {seed_scenario_3}")

# 【シナリオ4】コミュニティの中心（最高インフルエンサー）
internal_degrees = dict(sub_G_opp.degree())
top_influencer = max(internal_degrees, key=internal_degrees.get)
seed_scenario_4 = [top_influencer]
print(f"【シナリオ④】敵陣最高インフルエンサー: {seed_scenario_4} (内部次数: {internal_degrees[top_influencer]})")

# 【シナリオ5】コミュニティの端寄り（1つのユーザー）
periphery_candidates = [node for node, deg in sub_G_opp.degree() if deg <= 2]
random.seed(42)
seed_scenario_5 = [random.choice(periphery_candidates)]
print(f"【シナリオ⑤】敵陣の端(1ユーザー): {seed_scenario_5}")

# 【シナリオ6】コミュニティの端寄り（複数ユーザー）
num_seeds = 5
random.seed(42)
seed_scenario_6 = random.sample(periphery_candidates, min(num_seeds, len(periphery_candidates)))
print(f"【シナリオ⑥】敵陣の端(複数ユーザー {len(seed_scenario_6)}名): {seed_scenario_6}")

[Info] グラフ G を再生成しています...
--- 🚀 シナリオ別シードノード選出 (ターゲット: 2 vs 相手: 1) ---
【シナリオ①】ブリッジ(敵陣側ハブ): [2989]
【シナリオ②】ブリッジ(自陣側裏切り者): [4539]
【シナリオ③】ブリッジ(国境両側同時): [2989, 4539]
【シナリオ④】敵陣最高インフルエンサー: [2124] (内部次数: 18)
【シナリオ⑤】敵陣の端(1ユーザー): [3308]
【シナリオ⑥】敵陣の端(複数ユーザー 5名): [3308, 4360, 2157, 3560, 4636]
